# 1. Sửa tay các tags đã bị gắn sai

- Bước 1: Loại bỏ các tags "." thành tags "O"

- Bước 2: Kiểm tra và gán nhãn thủ công các tags "B" và "I" rỗng

- Bước 3: Kiểm tra và gán nhãn thủ công các tags "ORG", "PERSON", "VOLUME", "ASSET", "DATE", "MONEY", "RATE" với tiền tố "B-" và "I-"

In [1]:
import json
# Hàm kiểm tra các tags bị lỗi
file_paths = [
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl",
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl",
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl"
]
wrong_tags = [
    "B", "I", ".",
    "ORG", "PERSON", "VOLUME", 
    "ASSET", "DATE", "MONEY", "RATE"
]
for file_path in file_paths:
    file_name = file_path.split('/')[-1]
    print(f"Checking failed tags at {file_name}")
    
    for wrong_tag in wrong_tags:
        with open(file_path, "r", encoding="utf-8") as f:
            failed_list = []
            for idx, line in enumerate(f):
                data = json.loads(line)
                tags = data["tags"]

                # Chỉ cần có ORG là in index
                # CHỈ đúng khi có phần tử == "ORG"
                if wrong_tag in tags:
                    failed_list.append(idx+1)
            print(f'Number of wrong "{wrong_tag}" tags: {len(failed_list)}: {failed_list}')

Checking failed tags at train_raw.jsonl
Number of wrong "B" tags: 0: []
Number of wrong "I" tags: 0: []
Number of wrong "." tags: 0: []
Number of wrong "ORG" tags: 0: []
Number of wrong "PERSON" tags: 0: []
Number of wrong "VOLUME" tags: 0: []
Number of wrong "ASSET" tags: 0: []
Number of wrong "DATE" tags: 0: []
Number of wrong "MONEY" tags: 0: []
Number of wrong "RATE" tags: 0: []
Checking failed tags at dev_raw.jsonl
Number of wrong "B" tags: 0: []
Number of wrong "I" tags: 0: []
Number of wrong "." tags: 0: []
Number of wrong "ORG" tags: 0: []
Number of wrong "PERSON" tags: 0: []
Number of wrong "VOLUME" tags: 0: []
Number of wrong "ASSET" tags: 0: []
Number of wrong "DATE" tags: 0: []
Number of wrong "MONEY" tags: 0: []
Number of wrong "RATE" tags: 0: []
Checking failed tags at test_raw.jsonl
Number of wrong "B" tags: 0: []
Number of wrong "I" tags: 0: []
Number of wrong "." tags: 0: []
Number of wrong "ORG" tags: 0: []
Number of wrong "PERSON" tags: 0: []
Number of wrong "VOLUME"

# 2. Chuẩn hoá dấu câu

**Mục đích:**

- Loại bỏ tất cả trường hợp tách tokenization sai theo từng dấu câu. 

- Ví dụ 1: 

    + Trước: {"tokens": ["CIR", "vẫn", "được", "duy", "trì", "ở", "mức", "50%", "trong", "quý", "II."], "tags": ["O", "O", "O", "O", "O", "O", "O", "B-RATE", "O", "B-DATE", "I-DATE"]}

    + Sau: {"tokens": ["CIR", "vẫn", "được", "duy", "trì", "ở", "mức", "50%", "trong", "quý", "II", "."], "tags": ["O", "O", "O", "O", "O", "O", "O", "B-RATE", "O", "B-DATE", "I-DATE", "O"]}

- Ví dụ 2: 

    + Trước: {"tokens": ["Ngày", "21/1,", "Công", "ty", "TNHH", "Quản", "lý", "Quỹ", "SSI", "(", "SSIAM", ")",  "-", "thành", "viên", "thuộc", "CTCP", "Chứng", "khoán", "SSI", "(", "mã", "CK:", "SSI", ")", "công", "bố", "thành", "lập", "SSI", "Digital", "Ventures."], "tags": ["B-DATE", "I-DATE", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-ORG", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "O", "O", "B-ORG", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG"]}

    + Sau: {"tokens": ["Ngày", "21/1,", "Công", "ty", "TNHH", "Quản", "lý", "Quỹ", "SSI", "(", "SSIAM", ")", "-", "thành", "viên", "thuộc", "CTCP", "Chứng", "khoán", "SSI", "(", "mã", "CK", ":", "SSI", ")", "công", "bố", "thành", "lập", "SSI", "Digital", "Ventures."], "tags": ["B-DATE", "I-DATE", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-ORG", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "O", "O", "O", "B-ORG", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG"]}

- Ví dụ 3:

    + Trước: {"tokens": ["Một", "công", "ty", "trong", "danh", "mục", "đầu", "tư", "trước", "đây", "của", "Mekong", "Capital", "khi", "niêm", "yết", "cổ", "phiếu", "trên", "Sở", "Giao", "dịch", "Chứng", "khoán", "Thành", "phố", "Hồ", "Chí", "Minh", "(HOSE)", "đã", "chứng", "kiến", "giá", "cổ", "phiếu", "trong", "ngày", "giao", "dịch", "đầu", "tiên", "giảm", "20%."], "tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "B-ORG", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-RATE"]}

    + Sau: {"tokens": ["Một", "công", "ty", "trong", "danh", "mục", "đầu", "tư", "trước", "đây", "của", "Mekong", "Capital", "khi", "niêm", "yết", "cổ", "phiếu", "trên", "Sở", "Giao", "dịch", "Chứng", "khoán", "Thành", "phố", "Hồ", "Chí", "Minh", "(", "HOSE", ")", "đã", "chứng", "kiến", "giá", "cổ", "phiếu", "trong", "ngày", "giao", "dịch", "đầu", "tiên", "giảm", "20%."], "tags": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-ORG", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-RATE"]}

In [2]:
import json
import re

def is_number_with_dot(token):
    return bool(re.search(r'\d+\.\d+', token))

def split_token(token, tag):
    new_tokens = []
    new_tags = []

    # Nếu là số có dấu chấm → giữ nguyên
    if is_number_with_dot(token):
        return [token], [tag]

    original_tag = tag

    # 1. "(" ở đầu
    if token.startswith("("):
        new_tokens.append("(")
        new_tags.append("O")
        token = token[1:]

    # 2. kiểm tra "." ở cuối
    has_dot = token.endswith(".")

    if has_dot:
        token = token[:-1]

    # 3. kiểm tra ";" ở cuối
    has_semicolon = token.endswith(";")
    if has_semicolon:
        token = token[:-1]

    # 4. kiểm tra ")" ở cuối
    if token.endswith(")"):
        core = token[:-1]
        if core:
            new_tokens.append(core)
            new_tags.append(original_tag)
        new_tokens.append(")")
        new_tags.append("O")
    else:
        if token:
            new_tokens.append(token)
            new_tags.append(original_tag)

    # 5. thêm lại dấu ";" và "."
    if has_semicolon:
        new_tokens.append(";")
        new_tags.append("O")

    if has_dot:
        new_tokens.append(".")
        new_tags.append("O")

    return new_tokens, new_tags


def process_jsonl(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            item = json.loads(line)

            tokens = item["tokens"]
            tags = item["tags"]

            new_tokens = []
            new_tags = []

            for token, tag in zip(tokens, tags):
                tks, tgs = split_token(token, tag)
                new_tokens.extend(tks)
                new_tags.extend(tgs)

            assert len(new_tokens) == len(new_tags)

            item["tokens"] = new_tokens
            item["tags"] = new_tags

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
            
        file_name = output_path.split('/')[-1]
        print(f"Đã xử lý thành công, kết quả được lưu vào file {file_name}")

# ====== RUN ======
process_jsonl(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/train_raw.jsonl",
    output_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl"
)
process_jsonl(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/dev_raw.jsonl",
    output_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl"
)
process_jsonl(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/split/test_raw.jsonl",
    output_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl"
)

Đã xử lý thành công, kết quả được lưu vào file final_train_vifinner.jsonl
Đã xử lý thành công, kết quả được lưu vào file final_dev_vifinner.jsonl
Đã xử lý thành công, kết quả được lưu vào file final_test_vifinner.jsonl


# 3. Xoá ký tự lạ

- Xoá ký tự "?" lạ thông qua mã "\uFFFD" vì đây là ký tự không thể copy, không thể tìm bằng Cmd + F

In [3]:
import json
import os

BAD_CHAR = "\uFFFD"

def remove_bad_char(input_file, tmp_file):
    """
    Đây là code ghi đè lên input_file thông qua việc ghi vào file tạm (tmp_file)
    """
    with open(input_file, "r", encoding="utf-8") as fin, \
        open(tmp_file, "w", encoding="utf-8") as fout:

        for line in fin:
            item = json.loads(line)

            new_tokens = []
            new_tags = []

            for tok, tag in zip(item["tokens"], item["tags"]):
                if tok != BAD_CHAR:
                    new_tokens.append(tok)
                    new_tags.append(tag)

            item["tokens"] = new_tokens
            item["tags"] = new_tags

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
        # GHI ĐÈ AN TOÀN
        os.replace(tmp_file, input_file)    
    file_name = input_file.split('/')[-1]
    print(f"Đã xử lý thành công, kết quả được lưu vào file {file_name}")
    
remove_bad_char(
    input_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl",
    tmp_file   = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train.tmp.jsonl"
)
remove_bad_char(
    input_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl",
    tmp_file   = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev.tmp.jsonl"
)
remove_bad_char(
    input_file = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl",
    tmp_file   = "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test.tmp.jsonl"
)

Đã xử lý thành công, kết quả được lưu vào file final_train_vifinner.jsonl
Đã xử lý thành công, kết quả được lưu vào file final_dev_vifinner.jsonl
Đã xử lý thành công, kết quả được lưu vào file final_test_vifinner.jsonl


# 3. Gán nhãn đầy đủ cho tags thời gian

- Đối với các tags như ["năm", "2015"] hay ["ngày", "18/4/2023"], LLMs xảy ra bất đồng quan điểm việc gán tags " -DATE" cho "năm" hay "ngày" 
    
    --> Gán "B-DATE", đồng thời gán lại "I-DATE" cho các tokens phía sau.

In [4]:
import re

def fix_date_tags(tokens, tags):
    new_tags = tags.copy()

    time_keywords = {"ngày", "tháng", "năm", "quý"}

    def is_day(token):
        return bool(re.match(r'^\d{1,2}[/-]\d{1,2}[/-]\d{2,4},?$', token))

    def is_month(token):
        return token.isdigit() and 1 <= int(token) <= 12

    def is_year(token):
        return token.isdigit() and 1900 <= int(token) <= 2099

    def is_quarter(token):
        return token.upper() in {"I", "II", "III", "IV", "1", "2", "3", "4"}

    i = 0
    while i < len(tokens) - 1:
        cur = tokens[i].lower()
        nxt = tokens[i + 1]

        if (
            cur in time_keywords
            and new_tags[i] == "O"
            and new_tags[i + 1] == "B-DATE"
            and (
                is_day(nxt)
                or is_month(nxt)
                or is_year(nxt)
                or is_quarter(nxt)
            )
        ):
            new_tags[i] = "B-DATE"
            new_tags[i + 1] = "I-DATE"

            # kéo dài span: "Tháng 3 2024"
            j = i + 2
            while j < len(tokens) and new_tags[j] == "B-DATE":
                new_tags[j] = "I-DATE"
                j += 1

            i = j
        else:
            i += 1

    return new_tags


def process_jsonl_fix_date_safe(input_path):
    temp_path = input_path + ".tmp"

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(temp_path, "w", encoding="utf-8") as fout:

        for line in fin:
            item = json.loads(line)

            tokens = item["tokens"]
            tags = item["tags"]

            # FIX DATE TAGS
            item["tags"] = fix_date_tags(tokens, tags)

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")

    # Sau khi ghi xong mới replace
    os.replace(temp_path, input_path)

    print(f"✔ Đã xử lý xong & ghi đè an toàn: {input_path}")

# RUN       
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl"
)
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl"
)
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl"
)

✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl
✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl
✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl


# 4. Kiểm tra và chuẩn hoá tags "mồ côi"

- Kiểm tra các chuỗi tags có bắt đầu bằng "B-" và kết thúc bằng "I-"

- Gán nhãn lại theo cấu trúc "B-" - "I-"

- Ví dụ: 
    
    + Trước: ["O", "I-DATE", "I-DATE"] - Sau: ["O", "B-DATE", "I-DATE"]

    + Trước: ["O", "I-MONEY", "O"] - Sau: ["O", "B-MONEY", "O"]

In [5]:
def find_invalid_I_tags(tags):
    """
    Trả về list index của các tag I-XXX bị sai BIO
    """
    invalid_indices = []

    for i, tag in enumerate(tags):
        if tag.startswith("I-"):
            ent = tag[2:]  # XXX

            if i == 0:
                invalid_indices.append(i)
            else:
                prev = tags[i - 1]
                if prev not in (f"B-{ent}", f"I-{ent}"):
                    invalid_indices.append(i)

    return invalid_indices

def audit_bio_jsonl(path):
    file_name = path.split('/')[-1]
    print(f"Đang kiểm tra file {file_name}")
    total_bad = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            item = json.loads(line)
            bad = find_invalid_I_tags(item["tags"])
            if bad:
                total_bad.append(bad)
                print(f"Line {line_no}: lỗi BIO tại {bad}")
        if len(total_bad) == 0:
            print(f"  --> Không còn chuỗi BIO lỗi")
        else:
            print(f"Tổng số BIO lỗi: {len(total_bad)}")

audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl")
audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl")
audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl")

Đang kiểm tra file final_train_vifinner.jsonl
Line 60: lỗi BIO tại [15]
Line 131: lỗi BIO tại [22]
Line 196: lỗi BIO tại [16, 38]
Line 230: lỗi BIO tại [49]
Line 244: lỗi BIO tại [44, 47]
Line 256: lỗi BIO tại [11]
Line 262: lỗi BIO tại [9]
Line 265: lỗi BIO tại [11]
Line 268: lỗi BIO tại [11]
Line 296: lỗi BIO tại [17]
Line 385: lỗi BIO tại [17]
Line 429: lỗi BIO tại [36]
Line 596: lỗi BIO tại [13]
Line 639: lỗi BIO tại [10, 12]
Line 954: lỗi BIO tại [27]
Line 974: lỗi BIO tại [4]
Line 1343: lỗi BIO tại [3]
Line 1373: lỗi BIO tại [28]
Line 1392: lỗi BIO tại [8]
Line 1398: lỗi BIO tại [8]
Line 1414: lỗi BIO tại [15]
Line 1459: lỗi BIO tại [4]
Line 1460: lỗi BIO tại [25]
Line 1484: lỗi BIO tại [6]
Line 1511: lỗi BIO tại [14]
Line 1514: lỗi BIO tại [19]
Line 1566: lỗi BIO tại [15, 24]
Line 1607: lỗi BIO tại [8]
Line 1645: lỗi BIO tại [36]
Line 1652: lỗi BIO tại [7]
Line 1716: lỗi BIO tại [4]
Line 1723: lỗi BIO tại [13]
Line 1738: lỗi BIO tại [36]
Line 1753: lỗi BIO tại [13]
Line 2011: lỗ

In [6]:
def fix_orphan_I_tags(tags):
    """
    Auto-fix BIO:
    - I-X không có B-X trước → đổi thành B-X
    """
    fixed = tags.copy()

    for i, tag in enumerate(fixed):
        if tag.startswith("I-"):
            ent = tag[2:]

            if i == 0:
                fixed[i] = f"B-{ent}"
            else:
                prev = fixed[i - 1]
                if prev not in (f"B-{ent}", f"I-{ent}"):
                    fixed[i] = f"B-{ent}"

    return fixed

def process_jsonl_fix_date_safe(input_path):
    temp_path = input_path + ".tmp"

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(temp_path, "w", encoding="utf-8") as fout:

        for line in fin:
            item = json.loads(line)

            tokens = item["tokens"]
            tags = item["tags"]

            # FIX DATE TAGS
            item["tags"] = fix_orphan_I_tags(tags)

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")

    # Sau khi ghi xong mới replace
    os.replace(temp_path, input_path)

    print(f"✔ Đã xử lý xong & ghi đè an toàn: {input_path}")
            
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl"
)
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl"
)
process_jsonl_fix_date_safe(
    input_path="/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl"
)

✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl
✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl
✔ Đã xử lý xong & ghi đè an toàn: /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl


In [7]:
audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl")
audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/dev_vifinner.jsonl")
audit_bio_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/test_vifinner.jsonl")

Đang kiểm tra file train_vifinner.jsonl
  --> Không còn chuỗi BIO lỗi
Đang kiểm tra file dev_vifinner.jsonl
  --> Không còn chuỗi BIO lỗi
Đang kiểm tra file test_vifinner.jsonl
  --> Không còn chuỗi BIO lỗi


# 5. Sửa lỗi tag PERSON

In [8]:
def fix_title_person_tags(tokens, tags):
    titles = {"ông", "bà", "anh", "chị", "em"}
    n = len(tokens)

    i = 0
    while i < n - 1:
        token_lower = tokens[i].lower()

        # Nếu là từ xưng hô
        if token_lower in titles:
            # Nếu token sau là B-PERSON → sửa
            if tags[i + 1] == "B-PERSON":
                tags[i] = "B-PERSON"

                j = i + 1
                while j < n and tags[j] in {"B-PERSON", "I-PERSON"}:
                    tags[j] = "I-PERSON"
                    j += 1

                i = j
                continue

        i += 1

    return tokens, tags

def process_jsonl_file(input_path):
    temp_path = input_path + ".tmp"

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(temp_path, "w", encoding="utf-8") as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            sample = json.loads(line)

            tokens = sample.get("tokens")
            tags = sample.get("tags")

            if tokens is None or tags is None:
                fout.write(line + "\n")
                continue

            tokens, tags = fix_title_person_tags(tokens, tags)

            sample["tokens"] = tokens
            sample["tags"] = tags

            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")

    # Ghi đè file cũ
    os.replace(temp_path, input_path)

process_jsonl_file("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl")
process_jsonl_file("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl")
process_jsonl_file("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl")